# YOLO11 特徵模組

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/14-feature-module/)

先執行環境格，再閱讀、執行下方完整實驗。各節互相獨立，不需要上一節的 runtime。

本節在 CPU 逐節執行並保存輸出；合成資料短訓練、L4 與部署紀錄見網站驗證頁。真實場景長訓練仍待安排。這些範例沒有預訓練權重，也不需下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.2.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 逐節範例預設 CPU；L4 訓練與部署的實測另見網站驗證頁。保留相同版本的 CPU/CUDA build。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""C3k2-inspired split-transform-concatenate, not the full YOLO11 architecture."""
import torch
from torch import nn
from torch.nn import functional as F


class Bottleneck(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(),
                                    nn.Conv2d(channels, channels, 3, padding=1))

    def forward(self, x):
        return x + self.layers(x)


class SplitAggregate(nn.Module):
    def __init__(self, channels=8, hidden=4, blocks=2):
        super().__init__()
        self.project = nn.Conv2d(channels, 2 * hidden, 1)
        self.blocks = nn.ModuleList(Bottleneck(hidden) for _ in range(blocks))
        self.fuse = nn.Conv2d((2 + blocks) * hidden, channels, 1)

    def forward(self, x, inspect=False):
        a, b = self.project(x).chunk(2, dim=1)
        paths = [a, b]
        for block in self.blocks:
            paths.append(block(paths[-1]))
        joined = torch.cat(paths, dim=1)
        output = self.fuse(joined)
        return (output, paths, joined) if inspect else output


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    x = torch.randn(2, 8, 8, 8)
    plain = nn.Sequential(nn.Conv2d(8, 8, 3, padding=1), nn.ReLU(), nn.Conv2d(8, 8, 3, padding=1))
    module = SplitAggregate()
    optimizer = torch.optim.SGD(module.parameters(), lr=.02)
    target = x.roll(1, dims=-1)  # known local transformation; no detection-accuracy claim
    before = F.mse_loss(module(x), target).item()
    for _ in range(30):
        optimizer.zero_grad()
        loss = F.mse_loss(module(x), target)
        loss.backward()
        optimizer.step()
    after = F.mse_loss(module(x), target).item()
    assert module(x).shape == x.shape and after < before
    # Inspect direct concat slots separately from each path's total gradient.
    # b1's total gradient also includes its downstream use as the input to b2.
    optimizer.zero_grad()
    output, paths, joined = module(x, inspect=True)
    joined.retain_grad()
    for path in paths:
        path.retain_grad()
    F.mse_loss(output, target).backward()
    hidden = module.project.out_channels // 2
    assert len(paths) == 2 + len(module.blocks)
    for i, path in enumerate(paths):
        slot = slice(i * hidden, (i + 1) * hidden)
        assert torch.equal(joined[:, slot], path)
        assert path.grad is not None and path.grad.abs().sum() > 0
        assert joined.grad[:, slot].abs().sum() > 0
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in module.parameters())
    print('input/output:', tuple(x.shape), tuple(module(x).shape))
    channels = ' + '.join(str(hidden) for _ in paths)
    print(f'concatenated channels: {channels} = {module.fuse.in_channels}; fuse {module.fuse.in_channels} -> {module.fuse.out_channels}')
    print('parameters plain / split:', sum(p.numel() for p in plain.parameters()), sum(p.numel() for p in module.parameters()))
    print(f'local transformation MSE {before:.4f} -> {after:.4f}')
    print('concat order, each direct concat-slot gradient, and each path total gradient: verified')
    print('direct concat-slot gradient L1:', [round(float(joined.grad[:, i*hidden:(i+1)*hidden].abs().sum()), 4) for i in range(len(paths))])


if __name__ == '__main__':
    main()


input/output: (2, 8, 8, 8) (2, 8, 8, 8)
concatenated channels: 4 + 4 + 4 + 4 = 16; fuse 16 -> 8
parameters plain / split: 1168 800
local transformation MSE 1.0722 -> 1.0049
concat order, each direct concat-slot gradient, and each path total gradient: verified
direct concat-slot gradient L1: [0.2967, 0.3349, 0.3481, 0.2779]
